# TuQanes: evaluacion reproducible de los tres kernels en Quantinuum Nexus

Este es el punto de entrada unico para la fase Nexus. Mantiene intacta la generacion `last_version`, congela los tres ganadores por familia y separa de forma explicita: resultados exactos locales, previsualizaciones con shots y resultados descargados de Nexus.

**Seguridad:** el valor inicial `nexus_stage='local_only'` no autentica, no carga circuitos, no estima costos y no envia jobs. Las fases que consumen cuota requieren cambiar el stage y escribir una frase de reconocimiento exacta.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

def locate_module_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.extend([
            base,
            base / 'notebooks' / 'Johnny' / 'nexus_reproducible',
        ])
    for candidate in candidates:
        if (candidate / 'nexus_qsvm.py').exists():
            return candidate
    raise FileNotFoundError('No se encontro nexus_qsvm.py. Mantenga el notebook y el modulo en la misma carpeta.')

MODULE_DIR = locate_module_dir()
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from nexus_qsvm import (
    ExperimentConfig, LOCKED_MAPS, NEXUS_ACKNOWLEDGEMENT,
    aggregate_nexus_repeats, nexus_collect_map_results,
    nexus_compile_map_circuits, nexus_estimate_map_cost,
    nexus_load_execute_jobs, nexus_submit_map_repeats,
    nexus_upload_map_circuits, prepare_experiment,
    reconstruct_measured_kernels, run_local_analysis,
)

print(f'Modulo: {MODULE_DIR}')

## 1. Configuracion preregistrada

Use `subset_n=16` para el piloto. Cambie a 64 solamente despues de completar y revisar el piloto. `H2-1LE` es el primer backend recomendado porque aisla el error de shots; una ejecucion posterior con `H2-Emulator` debe usar otro `run_id`.

In [ ]:
CONFIG = ExperimentConfig(
    subset_n=64,
    preprocessing_protocol='global_median_external_pool',
    shots=512,
    repeats=3,
    backend_name='H2-Emulator',
    project_name='TuQanes-QSVM-Nexus',
    run_id='eval64-emulator-2maps-v1',
    nexus_stage='local_only',  # local_only | cost | submit | collect
    quota_acknowledgement='',  # requerido para cost/submit/collect
)
CONFIG.validate()

# Mapas que se ejecutan en Nexus: se excluye 'pauli_z_zz_linear_r1_robust'.
# No se toca LOCKED_MAPS para que la auditoria local siga validando los 3 candidatos.
SELECTED_MAPS = [m for m in LOCKED_MAPS if m != 'pauli_z_zz_linear_r1_robust']
print('Mapas para Nexus:', SELECTED_MAPS)
CONFIG

## 2. Preparacion y auditoria local

El preprocesador se ajusta con el development training excluyendo las 80 filas cuanticas. De esta forma no observa ninguna fila de evaluacion y un solo kernel medido puede reutilizar los cinco folds congelados.

In [ ]:
EXPERIMENT = prepare_experiment(CONFIG)
LOCAL_RESULTS = run_local_analysis(EXPERIMENT)

print(f'Artifacts: {EXPERIMENT.output_dir}')
display(Markdown('### Balance del subconjunto'))
display(EXPERIMENT.subset.groupby(['validation_fold', 'Potability']).size().unstack(fill_value=0))
display(Markdown('### Candidatos congelados desde la ejecucion anterior'))
display(LOCAL_RESULTS['candidate_selection'][[
    'map', 'family', 'C', 'f1_mean', 'balanced_accuracy_mean', 'mcc_mean'
]])

## 3. Comparacion local exacta sobre las mismas filas

La SVM-RBF evalua la grilla completa requerida, pero la fila comparativa usa `C=10` y `gamma=0.01`, congelados desde `quantum80_svm_cv_summary.csv`. Los tres valores `C` de QSVM tambien permanecen congelados; no se reajusta ningun hiperparametro con el piloto ni con resultados Nexus. En 16 muestras, estas metricas sirven para validar la tuberia y no para cambiar el ranking de mapas.

In [ ]:
classical_best = LOCAL_RESULTS['classical_locked']
classical_row = pd.DataFrame([{
    'model': 'SVM-RBF clasica (C/gamma congelados)',
    'map': '-',
    'C': classical_best['C'],
    'gamma': classical_best['gamma'],
    'f1_mean': classical_best['f1_mean'],
    'f1_std': classical_best['f1_std'],
    'balanced_accuracy_mean': classical_best['balanced_accuracy_mean'],
    'mcc_mean': classical_best['mcc_mean'],
}])
quantum_rows = LOCAL_RESULTS['qsvm_exact_summary'].copy()
quantum_rows.insert(0, 'model', 'QSVM exacta local')
quantum_rows['gamma'] = '-'
comparison = pd.concat([
    classical_row,
    quantum_rows[[
        'model', 'map', 'C', 'gamma', 'f1_mean', 'f1_std',
        'balanced_accuracy_mean', 'mcc_mean'
    ]],
], ignore_index=True)
display(comparison)
display(Markdown('**Auditoria:** la grilla completa queda en `classical_grid_summary.csv`; su mejor fila sobre este subconjunto no reemplaza la configuracion congelada.'))

## 4. Previsualizacion de shots y validacion del circuito

`local_binomial_preview` es una prueba local de reconstruccion y agregacion. No representa H2, Nexus ni ruido fisico.

In [ ]:
preview = LOCAL_RESULTS['shot_preview_summary']
preview_aggregate = (
    preview.groupby('map')
    .agg(
        repeats=('repeat', 'nunique'),
        f1_mean_across_repeats=('f1_mean', 'mean'),
        f1_std_across_repeats=('f1_mean', 'std'),
        balanced_accuracy_mean=('balanced_accuracy_mean', 'mean'),
        kernel_mae_mean=('kernel_mae_vs_exact', 'mean'),
    )
    .reset_index()
)
display(preview_aggregate)
print('Error maximo U(x_j)^dagger U(x_i) vs kernel exacto:',
      LOCAL_RESULTS['overlap_validation']['absolute_error'].max())
display(Markdown('### Carga prevista'))
display(LOCAL_RESULTS['workload'])

---
# Fases externas de Nexus

Las siguientes celdas permanecen inactivas en `local_only`. No llame `qnx.login()` dentro de Nexus Lab: la autenticacion es silenciosa. En un entorno externo, autentiquese manualmente antes de continuar. El reconocimiento de cuota no reemplaza la revision del portal y de `qnx.quotas`.

In [ ]:
if CONFIG.nexus_stage == 'local_only':
    print('Nexus desactivado: no se consultaron cuotas ni se transmitieron circuitos.')
else:
    if CONFIG.quota_acknowledgement != NEXUS_ACKNOWLEDGEMENT:
        raise RuntimeError('Revise los costos y escriba el reconocimiento de cuota exacto.')
    import qnexus as qnx
    display(qnx.quotas.get_all().df())

## 5. Carga y compilacion con checkpoints

Se ejecuta solamente en `cost` o `submit`. Las referencias se guardan localmente y se reutilizan al reanudar la sesion.

In [ ]:
REMOTE_REFS = {}
if CONFIG.nexus_stage in {'cost', 'submit'}:
    for map_name in SELECTED_MAPS:
        print(f'Cargando y compilando {map_name} ...')
        uploaded = nexus_upload_map_circuits(
            EXPERIMENT, map_name, LOCAL_RESULTS['pair_manifest']
        )
        compiled = nexus_compile_map_circuits(EXPERIMENT, map_name, uploaded)
        REMOTE_REFS[map_name] = {'uploaded': uploaded, 'compiled': compiled}
        print(f'{map_name}: {len(compiled)} circuitos compilados')
else:
    print('Carga y compilacion omitidas.')

## 6. Estimacion de costo

Esta celda solo corre en `cost`. La estimacion crea costing jobs y consume la cuota correspondiente. No envia ejecuciones H2.

In [ ]:
if CONFIG.nexus_stage == 'cost':
    cost_frames = []
    for map_name, refs in REMOTE_REFS.items():
        cost_frames.append(nexus_estimate_map_cost(
            EXPERIMENT, map_name, refs['compiled']
        ))
    COST_ESTIMATES = pd.concat(cost_frames, ignore_index=True)
    display(COST_ESTIMATES)
else:
    print('Costeo omitido. Use nexus_stage="cost" en una ejecucion deliberada.')

## 7. Envio de tres repeticiones

Esta celda solo corre en `submit`. Antes de cambiar el stage, revise `nexus/cost_estimates.csv`. Cada repeticion reutiliza los mismos circuitos compilados y crea jobs independientes.

In [ ]:
EXECUTE_JOBS = {}
if CONFIG.nexus_stage == 'submit':
    for map_name, refs in REMOTE_REFS.items():
        EXECUTE_JOBS[map_name] = nexus_submit_map_repeats(
            EXPERIMENT, map_name, refs['compiled']
        )
        print(map_name, len(EXECUTE_JOBS[map_name]), 'execute jobs enviados/checkpointed')
else:
    print('Envio omitido.')

## 8. Recoleccion, kernels crudos y agregacion

Use `collect` en una sesion posterior. Se descargan todos los counts, se reconstruye un kernel crudo por repeticion y se agregan las tres repeticiones sin filtrar la mejor.

In [ ]:
NEXUS_REPEAT_METRICS = pd.DataFrame()
if CONFIG.nexus_stage == 'collect':
    repeat_parts = []
    for map_name in SELECTED_MAPS:
        jobs = nexus_load_execute_jobs(EXPERIMENT, map_name)
        raw_counts = nexus_collect_map_results(
            EXPERIMENT, map_name, jobs, LOCAL_RESULTS['pair_manifest']
        )
        _, repeat_metrics = reconstruct_measured_kernels(
            EXPERIMENT, map_name, raw_counts
        )
        repeat_parts.append(repeat_metrics)
    NEXUS_REPEAT_METRICS = pd.concat(repeat_parts, ignore_index=True)
    NEXUS_AGGREGATE = aggregate_nexus_repeats(NEXUS_REPEAT_METRICS)
    aggregate_path = EXPERIMENT.output_dir / 'nexus' / 'aggregate_metrics.csv'
    NEXUS_AGGREGATE.to_csv(aggregate_path, index=False)
    display(NEXUS_AGGREGATE)
    print('Resumen guardado en:', aggregate_path)
else:
    print('Recoleccion omitida.')

## 9. Cierre

Antes de citar resultados, confirme que `run_manifest.json`, `raw_counts/*.csv`, los kernels crudos por repeticion, `aggregate_metrics.csv`, los IDs de jobs y los costos pertenecen al mismo `run_id` y backend. Mantenga separados `H2-1LE` y `H2-Emulator`.

In [ ]:
print('Run ID:', CONFIG.run_id)
print('Backend:', CONFIG.backend_name)
print('Stage:', CONFIG.nexus_stage)
print('Artifacts:', EXPERIMENT.output_dir)
print('Archivos locales principales:')
for path in sorted(EXPERIMENT.output_dir.rglob('*')):
    if path.is_file():
        print(' -', path.relative_to(EXPERIMENT.output_dir))